# LeetCode 123 — Best Time to Buy and Sell Stock III (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:

```

Đây là bài "mua bán cổ phiếu" phiên bản **tối đa 2 giao dịch**. Bài nghe giống bài Easy #121 (chỉ 1 giao dịch), nhưng chỉ thêm "giao dịch thứ hai" là các mẹo tham lam đơn giản đều **sai**. Ta sẽ đi từng bước, từ cách người mới tự nghĩ ra, tới lời giải gọn đến bất ngờ chỉ với **4 biến**.

---

## Sau notebook này, bạn sẽ nắm được

1. Ôn lại bài **#121 (1 giao dịch)** từ căn bản: brute force → "giá rẻ nhất đã thấy" (running min).
2. Vì sao các ý tưởng tham lam kiểu "lấy 2 đoạn tăng lớn nhất" lại **sai** (có phản ví dụ chạy được).
3. **Cách 1 — Chia mốc** `O(n²)`: giao dịch 1 nằm bên trái mốc, giao dịch 2 nằm bên phải mốc.
4. **Cách 2 — Prefix / Suffix** `O(n)`: tính sẵn `left[i]`, `right[i]` để khỏi tính lại.
5. **Cách 3 — DP trạng thái** `(ngày, số lượt mua còn lại, đang cầm hay không)`: từ đệ quy + memo tới bảng DP.
6. **Cách 4 — 4 biến `buy1, sell1, buy2, sell2`** `O(n)` thời gian, `O(1)` bộ nhớ — **cách chính**, kèm bảng trace trên `[3,3,5,0,0,3,1,4]` và ý nghĩa từng biến.
7. Lỗi hay gặp, mẹo nhận dạng, liên hệ **#188 (k giao dịch)** và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

You are given an array `prices` where `prices[i]` is the price of a given stock on the `i`-th day.

Find the maximum profit you can achieve. You may complete **at most two transactions**.

**Note:** You may not engage in multiple transactions simultaneously (i.e., you must sell the stock before you buy again).

Dịch nghĩa: cho mảng `prices`, `prices[i]` là giá cổ phiếu ngày thứ `i`. Hãy tìm **lợi nhuận lớn nhất**, được thực hiện **tối đa hai giao dịch** (một giao dịch = mua 1 lần rồi bán 1 lần). Không được giữ hai cổ phiếu cùng lúc: phải **bán xong rồi mới được mua lại**.

**Example 1**

```text
Input:  prices = [3,3,5,0,0,3,1,4]
Output: 6
Explanation: Buy on day 4 (price = 0) and sell on day 6 (price = 3), profit = 3-0 = 3.
Then buy on day 7 (price = 1) and sell on day 8 (price = 4), profit = 4-1 = 3.
```

**Example 2**

```text
Input:  prices = [1,2,3,4,5]
Output: 4
Explanation: Buy on day 1 (price = 1) and sell on day 5 (price = 5), profit = 5-1 = 4.
Note that you cannot buy on day 1, buy on day 2 and sell them later, as you are engaging
multiple transactions at the same time. You must sell before buying again.
```

**Example 3**

```text
Input:  prices = [7,6,4,3,1]
Output: 0
Explanation: In this case, no transaction is done, i.e. max profit = 0.
```

**Constraints**

- `1 <= prices.length <= 10^5`
- `0 <= prices[i] <= 10^5`

Ba điều cần để ý ngay:

- "**Tối đa** hai" nghĩa là được làm 0, 1 hoặc 2 giao dịch. Example 2 chỉ cần 1, Example 3 không làm giao dịch nào (lợi nhuận 0, **không bao giờ âm**).
- Đề đánh số ngày từ **1** ("day 4"), còn Python đánh index từ **0**: "day 4" chính là `prices[3]`.
- `n` tới `100 000` → thuật toán `O(n²)` (10¹⁰ bước) chắc chắn quá chậm. Ta cần `O(n)`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import
import random
import time

ex1 = [3, 3, 5, 0, 0, 3, 1, 4]
ex2 = [1, 2, 3, 4, 5]
ex3 = [7, 6, 4, 3, 1]

for name, p, expected in [("Example 1", ex1, 6), ("Example 2", ex2, 4), ("Example 3", ex3, 0)]:
    print(f"{name}: prices = {p} -> mong đợi {expected}")

# "day 4" của đề là index 3 trong Python
print("Giá ngày 4 (index 3) của example 1:", ex1[3])

Example 1: prices = [3, 3, 5, 0, 0, 3, 1, 4] -> mong đợi 6
Example 2: prices = [1, 2, 3, 4, 5] -> mong đợi 4
Example 3: prices = [7, 6, 4, 3, 1] -> mong đợi 0
Giá ngày 4 (index 3) của example 1: 0


## 2. Vẽ giá cổ phiếu ra cho dễ thấy

Với các bài mua bán cổ phiếu, hãy luôn **vẽ đồ thị giá**. Ta muốn: mua ở **đáy**, bán ở **đỉnh** ngay sau đó.

Cell dưới vẽ biểu đồ cột: mỗi cột là một ngày, cột cao bằng giá ngày đó. Bạn không cần hiểu chi tiết hàm vẽ, chỉ cần biết cách gọi `draw_prices(prices)`.

In [2]:
def draw_prices(prices):
    """Vẽ biểu đồ cột: mỗi ngày một cột, chiều cao = giá. Hàng dưới cùng là số ngày (đánh từ 1)."""
    top = max(prices)
    for level in range(top, 0, -1):
        row = "".join(" # " if p >= level else "   " for p in prices)
        print(f"{level:>3} |{row}")
    print("    +" + "---" * len(prices))
    print("ngày " + "".join(f"{d:^3}" for d in range(1, len(prices) + 1)))

draw_prices(ex1)

  5 |       #                
  4 |       #              # 
  3 | #  #  #        #     # 
  2 | #  #  #        #     # 
  1 | #  #  #        #  #  # 
    +------------------------
ngày  1  2  3  4  5  6  7  8 


### Tính tay Example 1 trước khi viết code

Nhìn đồ thị `[3,3,5,0,0,3,1,4]`:

- Có một "ngọn đồi" nhỏ: ngày 1–2 giá 3, ngày 3 giá 5 → mua 3 bán 5, lãi **2**.
- Ngày 4–5 giá rơi xuống **0** (đáy sâu nhất).
- Sau đó giá lên 3 (ngày 6), xuống 1 (ngày 7), lên 4 (ngày 8).

Thử vài kế hoạch:

| Kế hoạch | Giao dịch 1 | Giao dịch 2 | Tổng |
|---|---|---|---:|
| A | mua ngày 4 (0) → bán ngày 8 (4): **4** | không | 4 |
| B | mua ngày 1 (3) → bán ngày 3 (5): **2** | mua ngày 4 (0) → bán ngày 8 (4): **4** | **6** |
| C (đề) | mua ngày 4 (0) → bán ngày 6 (3): **3** | mua ngày 7 (1) → bán ngày 8 (4): **3** | **6** |
| D | mua ngày 1 (3) → bán ngày 3 (5): **2** | mua ngày 4 (0) → bán ngày 6 (3): **3** | 5 |

Nhận xét quan trọng:

1. Có **nhiều** kế hoạch cùng đạt tối ưu (B và C đều ra 6). Đề chỉ hỏi **con số** lợi nhuận, không hỏi kế hoạch.
2. Giao dịch đơn lẻ tốt nhất (A, lãi 4) **không nằm** trong kế hoạch C. Tức là "chọn giao dịch tốt nhất trước rồi tìm thêm" không phải lúc nào cũng dẫn đúng đường (ta sẽ thấy phản ví dụ rõ hơn ở mục 4).

## 3. Nền tảng: ôn lại bài #121 — chỉ **một** giao dịch

Trước khi giải 2 giao dịch, ta phải giải thật chắc bài 1 giao dịch, vì **mọi cách giải của bài 123 đều dùng lại nó**.

Bài #121: chọn một ngày mua `i` và một ngày bán `j` với `i <= j`, tối đa hoá `prices[j] - prices[i]`. Nếu không có lãi thì trả 0 (coi như `i == j`, mua và bán cùng ngày, lãi 0).

### 3.1. Cách người mới nghĩ ra: thử mọi cặp (brute force)

"Brute force" (vét cạn) nghĩa là thử **hết** mọi khả năng rồi lấy cái tốt nhất. Với 1 giao dịch: thử mọi cặp `(i, j)` với `i <= j`.

Ví dụ `[7,1,5,3,6,4]` (example của bài #121): cặp tốt nhất là mua giá 1 (index 1), bán giá 6 (index 4), lãi **5**.

In [3]:
def one_tx_bruteforce(prices: List[int]) -> int:
    """Bài 121 vét cạn: thử mọi cặp ngày mua i <= ngày bán j. O(n^2)."""
    best = 0
    n = len(prices)
    for i in range(n):            # ngày mua
        for j in range(i, n):     # ngày bán (không trước ngày mua)
            best = max(best, prices[j] - prices[i])
    return best

print(one_tx_bruteforce([7, 1, 5, 3, 6, 4]))   # 5
print(one_tx_bruteforce([7, 6, 4, 3, 1]))      # 0 (không có lãi -> không giao dịch)
print(one_tx_bruteforce(ex1))                  # 4 (mua 0, bán 4)

5
0
4


### 3.2. Cải tiến: "giá rẻ nhất đã thấy" (running min)

Câu hỏi then chốt: **nếu hôm nay (ngày `j`) mình bán, thì nên mua ngày nào?** Trả lời: ngày có giá **rẻ nhất** trong các ngày `0..j`. Không cần thử từng ngày mua!

Vậy chỉ cần đi từ trái sang phải và **nhớ giá thấp nhất từ đầu tới giờ**. Đây gọi là **running min** (giá trị nhỏ nhất "chạy" theo vòng lặp).

Cell khám phá: in `min_so_far` sau mỗi ngày. Để ý nó **chỉ giảm hoặc đứng yên**, không bao giờ tăng.

In [4]:
prices = [7, 1, 5, 3, 6, 4]
min_so_far = float("inf")        # "vô cùng": lớn hơn mọi số, để ngày đầu chắc chắn nhỏ hơn
for day, p in enumerate(prices):
    min_so_far = min(min_so_far, p)
    print(f"index {day}: giá {p}, rẻ nhất tới giờ = {min_so_far}")

index 0: giá 7, rẻ nhất tới giờ = 7
index 1: giá 1, rẻ nhất tới giờ = 1
index 2: giá 5, rẻ nhất tới giờ = 1
index 3: giá 3, rẻ nhất tới giờ = 1
index 4: giá 6, rẻ nhất tới giờ = 1
index 5: giá 4, rẻ nhất tới giờ = 1


### Kiến thức Python: `float("inf")` và `float("-inf")`

- `float("inf")` là **dương vô cùng**: lớn hơn mọi số. Dùng làm giá trị khởi đầu khi tìm **min**.
- `float("-inf")` là **âm vô cùng**: nhỏ hơn mọi số. Dùng làm giá trị khởi đầu khi tìm **max**, hoặc để đánh dấu "trạng thái chưa thể xảy ra".
- Cộng/trừ một số hữu hạn vào vô cùng thì vẫn là vô cùng: `float("-inf") + 5` vẫn là `-inf`. Nhờ vậy "trạng thái không thể" lan truyền mà không làm sai phép `max`.

Lát nữa ở cách 4, ta sẽ dùng `-inf` cho câu "chưa từng mua cổ phiếu nào".

In [5]:
NEG_INF = float("-inf")
print(float("inf") > 10**18)       # True
print(NEG_INF < -10**18)           # True
print(NEG_INF + 5)                 # -inf
print(max(NEG_INF, -3))            # -3  : max luôn "bỏ qua" -inf nếu có lựa chọn thật
print(max(0, NEG_INF + 7))         # 0

True
True
-inf
-3
0


### 3.3. Ghép lại: bài 121 trong `O(n)`

Mỗi ngày làm 2 việc **theo đúng thứ tự**:

1. Cập nhật `min_price = min(min_price, p)` (có thể chính hôm nay là ngày rẻ nhất).
2. Thử bán hôm nay: `best = max(best, p - min_price)`.

Vì bước 1 chạy trước, `p - min_price` luôn `>= 0` (tệ nhất là mua và bán cùng ngày, lãi 0).

Bảng trace trên `[7,1,5,3,6,4]`:

| index | giá | min_price | lãi nếu bán hôm nay | best |
|---:|---:|---:|---:|---:|
| 0 | 7 | 7 | 0 | 0 |
| 1 | 1 | 1 | 0 | 0 |
| 2 | 5 | 1 | 4 | 4 |
| 3 | 3 | 1 | 2 | 4 |
| 4 | 6 | 1 | **5** | **5** |
| 5 | 4 | 1 | 3 | 5 |

In [6]:
def one_tx(prices: List[int]) -> int:
    """Bài 121 tối ưu: O(n) thời gian, O(1) bộ nhớ."""
    min_price = float("inf")
    best = 0
    for p in prices:
        min_price = min(min_price, p)        # 1) giá mua rẻ nhất tới hôm nay
        best = max(best, p - min_price)      # 2) thử bán hôm nay
    return best


def trace_one_tx(prices):
    min_price, best = float("inf"), 0
    print("index | giá | min_price | lãi nếu bán | best")
    for i, p in enumerate(prices):
        min_price = min(min_price, p)
        best = max(best, p - min_price)
        print(f"{i:>5} | {p:>3} | {min_price:>9} | {p - min_price:>11} | {best:>4}")
    return best

trace_one_tx([7, 1, 5, 3, 6, 4])
assert one_tx([7, 1, 5, 3, 6, 4]) == one_tx_bruteforce([7, 1, 5, 3, 6, 4]) == 5
assert one_tx(ex3) == 0

index | giá | min_price | lãi nếu bán | best
    0 |   7 |         7 |           0 |    0
    1 |   1 |         1 |           0 |    0
    2 |   5 |         1 |           4 |    4
    3 |   3 |         1 |           2 |    4
    4 |   6 |         1 |           5 |    5
    5 |   4 |         1 |           3 |    5


## 4. Vì sao thêm "giao dịch thứ hai" lại khó? Hai ý tưởng tham lam **sai**

Người mới thường thử ngay một mẹo **tham lam** (greedy — mỗi bước chọn cái trông tốt nhất trước mắt). Ta thử hai mẹo phổ biến và kiểm tra chúng bằng một bộ vét cạn **chắc chắn đúng**.

**Mẹo sai A — "giao dịch tốt nhất, rồi thêm giao dịch tốt nhất ở phần còn lại".** Tìm giao dịch đơn tốt nhất `(i, j)`, rồi tìm thêm một giao dịch trong đoạn trước `i` hoặc sau `j`.

Phản ví dụ `[1,5,2,8]`: giao dịch đơn tốt nhất là mua 1 bán 8 (lãi 7) — nó "chiếm" cả mảng, không còn chỗ cho giao dịch thứ hai → tổng 7. Nhưng đáp án đúng là **(1→5) + (2→8) = 4 + 6 = 10**: tách đôi tốt hơn.

**Mẹo sai B — "cộng 2 đoạn tăng liên tiếp lớn nhất".** Chia mảng thành các đoạn đi lên liên tiếp, lấy 2 đoạn lãi nhiều nhất.

Phản ví dụ `[1,2,4,2,5,7,2,4,9,0]`: các đoạn đi lên là `1→4` (3), `2→7` (5), `2→9` (7). Hai đoạn lớn nhất: 5 + 7 = 12. Nhưng đáp án đúng là **13**: mua 1 bán 7 (lãi 6, **gộp** hai đoạn đầu, vượt qua chỗ giảm 4→2) + mua 2 bán 9 (lãi 7).

(Khi chạy cell dưới, bạn sẽ thấy mẹo B tình cờ đúng với `[1,5,2,8]` — nhưng chỉ cần **một** phản ví dụ là đủ để bác bỏ một mẹo.)

Bài học: lựa chọn giao dịch 1 **ảnh hưởng** tới giao dịch 2, nên phải xét chúng **cùng nhau**. Đó là dấu hiệu của **Dynamic Programming** (quy hoạch động) mà ta sẽ gặp ở cách 3, 4.

In [7]:
def max_profit_bruteforce(prices: List[int]) -> int:
    """Vét cạn THẬT SỰ: thử mọi i <= j <= k <= l (mua i, bán j, mua k, bán l). O(n^4).
    Chỉ dùng cho mảng rất nhỏ, làm 'trọng tài' để kiểm tra các cách khác."""
    n = len(prices)
    best = 0                                   # 0 giao dịch
    for i in range(n):
        for j in range(i, n):
            first = prices[j] - prices[i]
            best = max(best, first)            # 1 giao dịch
            for k in range(j, n):              # mua lại KHÔNG trước ngày bán j
                for l in range(k, n):
                    best = max(best, first + prices[l] - prices[k])   # 2 giao dịch
    return best


def greedy_wrong_B(prices):
    """Mẹo sai B: cộng 2 đoạn tăng liên tiếp lớn nhất."""
    gains, start = [], 0
    for i in range(1, len(prices) + 1):
        if i == len(prices) or prices[i] < prices[i - 1]:   # đoạn tăng kết thúc tại i-1
            gains.append(prices[i - 1] - prices[start])
            start = i
    gains.sort(reverse=True)
    return sum(gains[:2])


for p in ([1, 5, 2, 8], [1, 2, 4, 2, 5, 7, 2, 4, 9, 0]):
    print(p)
    print("   mẹo A (1 giao dịch tốt nhất chiếm hết):", one_tx(p))
    print("   mẹo B (2 đoạn tăng lớn nhất)          :", greedy_wrong_B(p))
    print("   vét cạn đúng                          :", max_profit_bruteforce(p))

[1, 5, 2, 8]
   mẹo A (1 giao dịch tốt nhất chiếm hết): 7
   mẹo B (2 đoạn tăng lớn nhất)          : 10
   vét cạn đúng                          : 10
[1, 2, 4, 2, 5, 7, 2, 4, 9, 0]
   mẹo A (1 giao dịch tốt nhất chiếm hết): 8
   mẹo B (2 đoạn tăng lớn nhất)          : 12
   vét cạn đúng                          : 13


## 5. Cách 1 — Chia mốc `k`: trái một giao dịch, phải một giao dịch

### Quan sát

Hai giao dịch không được chồng nhau: giao dịch 1 **bán xong** ở ngày `j`, giao dịch 2 **mua** ở ngày `k >= j`. Vậy luôn có một **vạch chia** (mốc) mà:

- giao dịch 1 nằm **hoàn toàn bên trái** mốc,
- giao dịch 2 nằm **hoàn toàn bên phải** mốc.

```text
prices:  3  3  5  0 | 0  3  1  4
         <- trái  ->|<-  phải  ->
         1 giao dịch | 1 giao dịch
```

Nếu biết mốc, bài toán tách thành **hai bài #121 độc lập**: tốt nhất bên trái + tốt nhất bên phải. Mà ta không biết mốc → **thử mọi mốc**.

Quy ước ở đây: mốc `k` cho đoạn trái là `prices[0..k]` và đoạn phải là `prices[k..n-1]` (ngày `k` thuộc cả hai bên). Dùng chung ngày `k` có sao không? Không sao: nếu bên trái bán ngày `k` và bên phải mua ngày `k`, tổng lãi `(p_k - a) + (b - p_k) = b - a` — đúng bằng một giao dịch "mua a bán b", tức vẫn là một kế hoạch hợp lệ.

Tại sao không bỏ sót? Kế hoạch 0 hay 1 giao dịch: một bên lãi 0 là được. Kế hoạch 2 giao dịch (bán lần 1 ở ngày `j`): chọn mốc `k = j` là nó được tính.

### Kiến thức Python: slicing `prices[:k+1]` và `prices[k:]`

`a[start:end]` lấy các phần tử từ index `start` tới **trước** `end`. Bỏ trống `start` = từ đầu, bỏ trống `end` = tới hết.

In [8]:
p = [3, 3, 5, 0, 0, 3, 1, 4]
k = 3
print("trái  p[:k+1] =", p[:k + 1])   # index 0..3
print("phải  p[k:]   =", p[k:])       # index 3..7  (ngày k nằm ở cả hai)
print("mốc k=0:", p[:1], p[0:])
print("mốc k=7:", p[:8], p[7:])

trái  p[:k+1] = [3, 3, 5, 0]
phải  p[k:]   = [0, 0, 3, 1, 4]
mốc k=0: [3] [3, 3, 5, 0, 0, 3, 1, 4]
mốc k=7: [3, 3, 5, 0, 0, 3, 1, 4] [4]


In [9]:
def max_profit_split(prices: List[int]) -> int:
    """Cách 1: thử mọi mốc k, mỗi bên giải bài 121. O(n^2) thời gian, O(n) bộ nhớ (do slicing)."""
    best = 0
    for k in range(len(prices)):
        best = max(best, one_tx(prices[:k + 1]) + one_tx(prices[k:]))
    return best

for p, expected in [(ex1, 6), (ex2, 4), (ex3, 0), ([1, 5, 2, 8], 10)]:
    print(p, "->", max_profit_split(p), "(mong đợi", expected, ")")

[3, 3, 5, 0, 0, 3, 1, 4] -> 6 (mong đợi 6 )
[1, 2, 3, 4, 5] -> 4 (mong đợi 4 )
[7, 6, 4, 3, 1] -> 0 (mong đợi 0 )
[1, 5, 2, 8] -> 10 (mong đợi 10 )


### Trace cách 1 trên Example 1

| k | trái `prices[0..k]` | best trái | phải `prices[k..7]` | best phải | tổng |
|---:|---|---:|---|---:|---:|
| 0 | `[3]` | 0 | `[3,3,5,0,0,3,1,4]` | 4 | 4 |
| 1 | `[3,3]` | 0 | `[3,5,0,0,3,1,4]` | 4 | 4 |
| 2 | `[3,3,5]` | 2 | `[5,0,0,3,1,4]` | 4 | **6** |
| 3 | `[3,3,5,0]` | 2 | `[0,0,3,1,4]` | 4 | **6** |
| 4 | `[3,3,5,0,0]` | 2 | `[0,3,1,4]` | 4 | **6** |
| 5 | `[3,3,5,0,0,3]` | 3 | `[3,1,4]` | 3 | **6** |
| 6 | `[3,3,5,0,0,3,1]` | 3 | `[1,4]` | 3 | **6** |
| 7 | `[3,3,5,0,0,3,1,4]` | 4 | `[4]` | 0 | 4 |

- Mốc `k = 2..4` ứng với kế hoạch B (3→5, rồi 0→4).
- Mốc `k = 5, 6` ứng với kế hoạch C của đề (0→3, rồi 1→4).

In [10]:
def trace_split(prices):
    best = 0
    for k in range(len(prices)):
        left, right = prices[:k + 1], prices[k:]
        a, b = one_tx(left), one_tx(right)
        best = max(best, a + b)
        print(f"k={k}: trái={str(left):<26} best={a} | phải={str(right):<26} best={b} | tổng={a + b}")
    print("Đáp án:", best)

trace_split(ex1)

k=0: trái=[3]                        best=0 | phải=[3, 3, 5, 0, 0, 3, 1, 4]   best=4 | tổng=4
k=1: trái=[3, 3]                     best=0 | phải=[3, 5, 0, 0, 3, 1, 4]      best=4 | tổng=4
k=2: trái=[3, 3, 5]                  best=2 | phải=[5, 0, 0, 3, 1, 4]         best=4 | tổng=6
k=3: trái=[3, 3, 5, 0]               best=2 | phải=[0, 0, 3, 1, 4]            best=4 | tổng=6
k=4: trái=[3, 3, 5, 0, 0]            best=2 | phải=[0, 3, 1, 4]               best=4 | tổng=6
k=5: trái=[3, 3, 5, 0, 0, 3]         best=3 | phải=[3, 1, 4]                  best=3 | tổng=6
k=6: trái=[3, 3, 5, 0, 0, 3, 1]      best=3 | phải=[1, 4]                     best=3 | tổng=6
k=7: trái=[3, 3, 5, 0, 0, 3, 1, 4]   best=4 | phải=[4]                        best=0 | tổng=4
Đáp án: 6


### Cách 1 vướng ở đâu?

Với mỗi mốc `k`, ta gọi `one_tx` trên hai đoạn có tổng độ dài `n + 1` → mỗi mốc tốn `O(n)`, có `n` mốc → tổng `O(n²)`.

- `n = 10^5` → khoảng `10^10` bước: Python chạy hàng giờ.
- Điểm phí phạm: khi đi từ mốc `k` sang `k+1`, đoạn trái chỉ **thêm một ngày**, nhưng ta lại tính `one_tx` **từ đầu**. Đoạn phải chỉ **bớt một ngày**, cũng tính lại từ đầu.

Cell dưới đo thử với `n = 2000` để thấy thời gian tăng theo bình phương (con số cụ thể phụ thuộc máy).

In [11]:
random.seed(0)
for n in (500, 1000, 2000):
    arr = [random.randint(0, 10**5) for _ in range(n)]
    t0 = time.perf_counter()
    max_profit_split(arr)
    print(f"n={n:>5}: {time.perf_counter() - t0:.3f} giây")
print("n tăng gấp đôi -> thời gian tăng khoảng gấp bốn: dấu hiệu O(n^2)")

n=  500: 0.013 giây
n= 1000: 0.055 giây


n= 2000: 0.240 giây
n tăng gấp đôi -> thời gian tăng khoảng gấp bốn: dấu hiệu O(n^2)


## 6. Cách 2 — Prefix / Suffix: tính sẵn để khỏi tính lại

### Kiến thức nền: mảng tiền tố (prefix array)

**Prefix array** (mảng tiền tố) là mảng mà phần tử thứ `i` tóm tắt **toàn bộ đoạn `0..i`**. Ví dụ quen thuộc:

- prefix sum: `ps[i] = a[0] + ... + a[i]`, tính bằng `ps[i] = ps[i-1] + a[i]`.
- prefix max: `pm[i] = max(a[0..i])`, tính bằng `pm[i] = max(pm[i-1], a[i])`.

Điểm mấu chốt: phần tử mới chỉ cần **phần tử trước đó** + **một ô dữ liệu mới**, không phải duyệt lại từ đầu. Tương tự, **suffix array** (mảng hậu tố) tóm tắt đoạn `i..n-1` và được tính bằng cách đi **từ phải sang trái**.

In [12]:
a = [3, 1, 4, 1, 5]
ps, pm = [a[0]], [a[0]]
for i in range(1, len(a)):
    ps.append(ps[-1] + a[i])
    pm.append(max(pm[-1], a[i]))
print("a       =", a)
print("prefix sum =", ps)   # [3, 4, 8, 9, 14]
print("prefix max =", pm)   # [3, 3, 4, 4, 5]

sm = [0] * len(a)            # suffix max: đi từ phải sang trái
sm[-1] = a[-1]
for i in range(len(a) - 2, -1, -1):
    sm[i] = max(sm[i + 1], a[i])
print("suffix max =", sm)   # [5, 5, 5, 5, 5]

a       = [3, 1, 4, 1, 5]
prefix sum = [3, 4, 8, 9, 14]
prefix max = [3, 3, 4, 4, 5]
suffix max = [5, 5, 5, 5, 5]


### Áp dụng: `left[i]` và `right[i]`

Định nghĩa:

- `left[i]` = lãi tốt nhất của **một** giao dịch trong `prices[0..i]` (bài 121 trên đoạn trái).
- `right[i]` = lãi tốt nhất của **một** giao dịch trong `prices[i..n-1]` (bài 121 trên đoạn phải).

Khi đó cách 1 chỉ là: `đáp án = max(left[k] + right[k])` với mọi `k`.

**Tính `left`** (đi xuôi) — y hệt bài 121, chỉ là lưu `best` lại ở mỗi bước:

```python
min_price = min(min_price, prices[i])
left[i] = max(left[i-1], prices[i] - min_price)      # hoặc không bán hôm nay, hoặc bán hôm nay
```

**Tính `right`** (đi ngược) — đối xứng: đứng ở ngày `i` và hỏi "nếu **mua** hôm nay thì nên **bán** ngày nào?" → ngày có giá **cao nhất** ở bên phải:

```python
max_price = max(max_price, prices[i])
right[i] = max(right[i+1], max_price - prices[i])    # hoặc không mua hôm nay, hoặc mua hôm nay
```

Bảng cho Example 1:

| index i | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| prices | 3 | 3 | 5 | 0 | 0 | 3 | 1 | 4 |
| left[i] | 0 | 0 | 2 | 2 | 2 | 3 | 3 | 4 |
| right[i] | 4 | 4 | 4 | 4 | 4 | 3 | 3 | 0 |
| left + right | 4 | 4 | **6** | **6** | **6** | **6** | **6** | 4 |

So với bảng trace cách 1: cột "tổng" **giống hệt**. Cách 2 chính là cách 1 nhưng **nhớ lại** kết quả thay vì tính lại.

In [13]:
def build_left(prices):
    n = len(prices)
    left = [0] * n
    min_price = prices[0]
    for i in range(1, n):
        min_price = min(min_price, prices[i])
        left[i] = max(left[i - 1], prices[i] - min_price)
    return left


def build_right(prices):
    n = len(prices)
    right = [0] * n
    max_price = prices[-1]
    for i in range(n - 2, -1, -1):          # đi từ phải sang trái
        max_price = max(max_price, prices[i])
        right[i] = max(right[i + 1], max_price - prices[i])
    return right


L, R = build_left(ex1), build_right(ex1)
print("prices      :", ex1)
print("left        :", L)
print("right       :", R)
print("left + right:", [x + y for x, y in zip(L, R)])

prices      : [3, 3, 5, 0, 0, 3, 1, 4]
left        : [0, 0, 2, 2, 2, 3, 3, 4]
right       : [4, 4, 4, 4, 4, 3, 3, 0]
left + right: [4, 4, 6, 6, 6, 6, 6, 4]


In [14]:
def max_profit_prefix_suffix(prices: List[int]) -> int:
    """Cách 2: O(n) thời gian, O(n) bộ nhớ."""
    left = build_left(prices)
    right = build_right(prices)
    return max(left[k] + right[k] for k in range(len(prices)))

for p, expected in [(ex1, 6), (ex2, 4), (ex3, 0), ([1, 5, 2, 8], 10), ([1, 2, 4, 2, 5, 7, 2, 4, 9, 0], 13)]:
    got = max_profit_prefix_suffix(p)
    print(p, "->", got)
    assert got == expected

[3, 3, 5, 0, 0, 3, 1, 4] -> 6
[1, 2, 3, 4, 5] -> 4
[7, 6, 4, 3, 1] -> 0
[1, 5, 2, 8] -> 10
[1, 2, 4, 2, 5, 7, 2, 4, 9, 0] -> 13


### Độ phức tạp và giới hạn của cách 2

Gọi `n = len(prices)`.

- Thời gian: `O(n)` — ba lượt duyệt (dựng `left`, dựng `right`, lấy max).
- Bộ nhớ: `O(n)` — hai mảng `left`, `right`.

Cách 2 **đã đủ nhanh để nộp**. Nhưng nó còn hai điểm chưa đẹp:

1. Tốn `O(n)` bộ nhớ phụ.
2. Ý tưởng "chia mốc" chỉ hợp với **đúng 2** giao dịch. Với `k` giao dịch (bài #188), ta phải chia `k-1` mốc — không mở rộng tự nhiên được.

Để có lời giải **tổng quát hơn** và **ít bộ nhớ hơn**, ta đổi góc nhìn: mô tả bài toán bằng **trạng thái** của người chơi chứng khoán ở mỗi ngày.

## 7. Cách 3 — DP trạng thái: (ngày, số lượt mua còn lại, đang cầm hay không)

### 7.1. Kiến thức nền: đệ quy (recursion)

**Đệ quy** là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn, cho tới khi gặp **trường hợp cơ sở** (base case) có đáp án ngay.

Ví dụ mini: tổng các phần tử từ index `i` tới cuối = `a[i]` + tổng từ `i+1` tới cuối. Base case: `i == len(a)` thì tổng là 0.

In [15]:
def suffix_sum(a, i):
    if i == len(a):              # base case: hết mảng
        return 0
    return a[i] + suffix_sum(a, i + 1)   # bài nhỏ hơn: bắt đầu từ i+1

print(suffix_sum([3, 1, 4], 0))   # 3 + (1 + (4 + 0)) = 8

8


### 7.2. Kiến thức nền: memoization (ghi nhớ kết quả)

Đệ quy có thể **gọi lại cùng một bài con rất nhiều lần**. Ví dụ Fibonacci: `fib(n) = fib(n-1) + fib(n-2)`. Để tính `fib(5)`, ta tính `fib(3)` hai lần, `fib(2)` ba lần... số lời gọi tăng theo **cấp số mũ**.

**Memoization** (memo): lần đầu tính xong bài con nào thì **ghi vào sổ** (một `dict`); lần sau gặp lại thì tra sổ, khỏi tính.

Python có sẵn "cuốn sổ" này: decorator `@lru_cache(maxsize=None)` từ module `functools`. Đặt nó ngay trên `def` là hàm tự nhớ kết quả theo **bộ tham số**. (Decorator là cú pháp `@ten` đặt trên hàm để "bọc thêm" tính năng cho hàm đó.)

In [16]:
from functools import lru_cache

calls = 0
def fib_naive(n):
    global calls
    calls += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

print("fib_naive(20) =", fib_naive(20), "| số lời gọi:", calls)

memo = {}                                  # memo tự làm bằng dict
def fib_memo(n):
    if n in memo:
        return memo[n]
    memo[n] = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]
print("fib_memo(20)  =", fib_memo(20), "| số ô trong sổ:", len(memo))

@lru_cache(maxsize=None)                   # memo có sẵn của Python
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)
print("fib_cache(20) =", fib_cache(20), "|", fib_cache.cache_info())

fib_naive(20) = 6765 | số lời gọi: 21891
fib_memo(20)  = 6765 | số ô trong sổ: 21
fib_cache(20) = 6765 | CacheInfo(hits=18, misses=21, maxsize=None, currsize=21)


### 7.3. Mô hình hoá bài toán bằng trạng thái

Mỗi sáng, người chơi đứng ở **ngày `i`** và chỉ cần biết hai điều để ra quyết định:

- `k` = còn được **mua** bao nhiêu lần nữa (ban đầu `k = 2`; mỗi lần mua thì `k` giảm 1),
- `holding` = có đang **cầm** cổ phiếu không (`True`/`False`).

Bộ ba `(i, k, holding)` gọi là **trạng thái** (state). Định nghĩa:

> `best(i, k, holding)` = lợi nhuận **thêm được** tối đa từ ngày `i` tới hết, khi đang ở trạng thái đó.

Ở ngày `i` có các lựa chọn:

| Đang ở | Lựa chọn | Lợi nhuận hôm nay | Sang trạng thái |
|---|---|---:|---|
| bất kỳ | **nghỉ** (không làm gì) | 0 | `(i+1, k, holding)` |
| `holding = True` | **bán** | `+prices[i]` | `(i+1, k, False)` |
| `holding = False`, `k > 0` | **mua** | `-prices[i]` | `(i+1, k-1, True)` |

Base case: `i == n` (hết ngày) → không làm thêm được gì, trả 0. Nếu lúc đó vẫn đang cầm cổ phiếu thì coi như cổ phiếu bỏ đi (giá trị 0) — tiền mua đã bị trừ lúc mua rồi, nên kế hoạch kiểu này luôn **tệ hơn** việc không mua, không bao giờ được chọn làm đáp án.

Đáp án: `best(0, 2, False)` — ngày đầu, còn 2 lượt mua, tay trắng.

Ví dụ nhỏ `[3,5]`: từ `(0,2,False)` có thể nghỉ → `(1,2,False)` hoặc mua giá 3 → `(1,1,True)`. Từ `(1,1,True)` bán giá 5 được `+5`. Tổng `-3 + 5 = 2`. Đúng!

In [17]:
def max_profit_recursive_naive(prices, K=2):
    """Đệ quy CHƯA memo, đếm số lời gọi để thấy nó bùng nổ."""
    n = len(prices)
    counter = [0]

    def best(i, k, holding):
        counter[0] += 1
        if i == n:
            return 0
        result = best(i + 1, k, holding)                                  # nghỉ
        if holding:
            result = max(result, prices[i] + best(i + 1, k, False))       # bán
        elif k > 0:
            result = max(result, -prices[i] + best(i + 1, k - 1, True))   # mua
        return result

    return best(0, K, False), counter[0]

for p in ([3, 5], [3, 5, 1, 4], ex1, ex1 + ex1):
    ans, cnt = max_profit_recursive_naive(p)
    print(f"n={len(p):>2}: đáp án={ans:>2}, số lời gọi={cnt}")

n= 2: đáp án= 2, số lời gọi=7
n= 4: đáp án= 5, số lời gọi=31
n= 8: đáp án= 6, số lời gọi=381
n=16: đáp án= 9, số lời gọi=9401


Số lời gọi tăng rất nhanh khi `n` tăng (cùng một trạng thái `(i, k, holding)` bị tính lại nhiều lần). Nhưng tổng số trạng thái **khác nhau** chỉ có `(n + 1) × 3 × 2` (vì `k` thuộc `{0,1,2}`, `holding` thuộc `{False, True}`). Thêm `@lru_cache` là mỗi trạng thái chỉ được tính **một lần**.

In [18]:
def max_profit_memo(prices: List[int], K: int = 2) -> int:
    """Cách 3a: đệ quy + memo. O(n*K) thời gian và bộ nhớ.
    Chỉ để HỌC: với n lớn sẽ vượt giới hạn độ sâu đệ quy của Python."""
    n = len(prices)

    @lru_cache(maxsize=None)
    def best(i, k, holding):
        if i == n:
            return 0
        result = best(i + 1, k, holding)                                  # nghỉ
        if holding:
            result = max(result, prices[i] + best(i + 1, k, False))       # bán
        elif k > 0:
            result = max(result, -prices[i] + best(i + 1, k - 1, True))   # mua
        return result

    ans = best(0, K, False)
    print(f"   (n={n}: số trạng thái thực sự được tính = {best.cache_info().currsize})")
    return ans

for p, expected in [(ex1, 6), (ex2, 4), (ex3, 0), (ex1 + ex1, None)]:
    print(p, "->", max_profit_memo(p))

   (n=8: số trạng thái thực sự được tính = 35)
[3, 3, 5, 0, 0, 3, 1, 4] -> 6
   (n=5: số trạng thái thực sự được tính = 20)
[1, 2, 3, 4, 5] -> 4
   (n=5: số trạng thái thực sự được tính = 20)
[7, 6, 4, 3, 1] -> 0
   (n=16: số trạng thái thực sự được tính = 75)
[3, 3, 5, 0, 0, 3, 1, 4, 3, 3, 5, 0, 0, 3, 1, 4] -> 9


### 7.4. Từ memo sang bảng DP (bottom-up)

Memo là "từ trên xuống" (top-down): hỏi `best(0, 2, False)`, đệ quy tự đi xuống các ngày sau. Nhược điểm: độ sâu đệ quy bằng `n`, mà Python mặc định chỉ cho khoảng 1000 tầng (Jupyter thường nâng lên 3000) → `n = 10^5` sẽ gặp `RecursionError`.

**Bảng DP** (bottom-up, "từ dưới lên"): tự điền kết quả vào một mảng theo thứ tự sao cho khi cần ô nào thì ô đó **đã có sẵn**. Vì `best(i, ...)` cần `best(i+1, ...)`, ta điền **từ ngày cuối lùi về ngày đầu**:

```text
dp[i][k][h]  với  i = n, n-1, ..., 0;   k = 0..K;   h = 0 (tay trắng) / 1 (đang cầm)

dp[n][k][h] = 0
dp[i][k][1] = max(dp[i+1][k][1],              prices[i] + dp[i+1][k][0])     # nghỉ / bán
dp[i][k][0] = max(dp[i+1][k][0],             -prices[i] + dp[i+1][k-1][1])   # nghỉ / mua (k > 0)
```

### Kiến thức Python: tạo list nhiều chiều cho đúng

Muốn bảng `(n+1) × (K+1) × 2` toàn số 0, phải dùng **list comprehension** lồng nhau. Đừng dùng `[[0, 0]] * 3`: phép `*` **nhân bản tham chiếu**, các hàng là **cùng một list**, sửa một hàng là tất cả đổi theo.

In [19]:
bad = [[0, 0]] * 3
bad[0][0] = 9
print("SAI :", bad)          # [[9, 0], [9, 0], [9, 0]]  <- cả 3 hàng đổi!

good = [[0, 0] for _ in range(3)]
good[0][0] = 9
print("ĐÚNG:", good)         # [[9, 0], [0, 0], [0, 0]]

n, K = 2, 2
table = [[[0, 0] for _ in range(K + 1)] for _ in range(n + 1)]
print("bảng 3 chiều (n+1) x (K+1) x 2:", table)

SAI : [[9, 0], [9, 0], [9, 0]]
ĐÚNG: [[9, 0], [0, 0], [0, 0]]
bảng 3 chiều (n+1) x (K+1) x 2: [[[0, 0], [0, 0], [0, 0]], [[0, 0], [0, 0], [0, 0]], [[0, 0], [0, 0], [0, 0]]]


In [20]:
def max_profit_dp_table(prices: List[int], K: int = 2) -> int:
    """Cách 3b: bảng DP bottom-up. O(n*K) thời gian, O(n*K) bộ nhớ.
    Tham số K cho phép giải luôn bài 'tối đa K giao dịch' (#188)."""
    n = len(prices)
    dp = [[[0, 0] for _ in range(K + 1)] for _ in range(n + 1)]   # dp[n][*][*] = 0
    for i in range(n - 1, -1, -1):              # lùi từ ngày cuối về ngày đầu
        for k in range(K + 1):
            # đang cầm: nghỉ, hoặc bán hôm nay
            dp[i][k][1] = max(dp[i + 1][k][1], prices[i] + dp[i + 1][k][0])
            # tay trắng: nghỉ, hoặc (nếu còn lượt) mua hôm nay
            dp[i][k][0] = dp[i + 1][k][0]
            if k > 0:
                dp[i][k][0] = max(dp[i][k][0], -prices[i] + dp[i + 1][k - 1][1])
    return dp[0][K][0]

for p, expected in [(ex1, 6), (ex2, 4), (ex3, 0), ([1, 5, 2, 8], 10)]:
    assert max_profit_dp_table(p) == expected
print("max_profit_dp_table: PASS các ví dụ")
print("Cùng bảng đó, nhưng chỉ 1 giao dịch (K=1):", max_profit_dp_table(ex1, K=1))   # 4 = bài 121

max_profit_dp_table: PASS các ví dụ
Cùng bảng đó, nhưng chỉ 1 giao dịch (K=1): 4


### Trace bảng DP trên `[3,5,1,4]`

Mỗi ô ghi dạng `(tay trắng, đang cầm)` = `(dp[i][k][0], dp[i][k][1])`. Điền từ dưới (`i = 4`) lên trên (`i = 0`):

| i | giá | k = 0 | k = 1 | k = 2 |
|---:|---:|---|---|---|
| 4 | — | (0, 0) | (0, 0) | (0, 0) |
| 3 | 4 | (0, 4) | (0, 4) | (0, 4) |
| 2 | 1 | (0, 4) | (3, 4) | (3, 4) |
| 1 | 5 | (0, 5) | (3, 8) | (3, 8) |
| 0 | 3 | (0, 5) | (3, 8) | (**5**, 8) |

Đọc vài ô:

- `dp[3][k][1] = 4`: đang cầm vào ngày cuối (giá 4) → bán luôn, được 4.
- `dp[2][1][0] = 3`: tay trắng ở ngày giá 1, còn 1 lượt → mua 1 bán 4, lãi 3.
- `dp[0][2][0] = 5`: đáp án = (mua 3 bán 5) + (mua 1 bán 4) = 2 + 3.
- Cột `k = 0` ở trạng thái tay trắng luôn là 0: hết lượt mua thì không kiếm thêm được gì.
- Ô "đang cầm" ở `k = 2` thực ra không bao giờ đạt tới (đã mua thì `k <= 1`), nhưng tính thừa cũng vô hại.

In [21]:
def trace_dp_table(prices, K=2):
    n = len(prices)
    dp = [[[0, 0] for _ in range(K + 1)] for _ in range(n + 1)]
    print(f"i={n} | giá=- | " + "  ".join(f"k={k}:{tuple(dp[n][k])}" for k in range(K + 1)))
    for i in range(n - 1, -1, -1):
        for k in range(K + 1):
            dp[i][k][1] = max(dp[i + 1][k][1], prices[i] + dp[i + 1][k][0])
            dp[i][k][0] = dp[i + 1][k][0]
            if k > 0:
                dp[i][k][0] = max(dp[i][k][0], -prices[i] + dp[i + 1][k - 1][1])
        print(f"i={i} | giá={prices[i]} | " + "  ".join(f"k={k}:{tuple(dp[i][k])}" for k in range(K + 1)))
    print("Đáp án dp[0][K][0] =", dp[0][K][0])

trace_dp_table([3, 5, 1, 4])

i=4 | giá=- | k=0:(0, 0)  k=1:(0, 0)  k=2:(0, 0)
i=3 | giá=4 | k=0:(0, 4)  k=1:(0, 4)  k=2:(0, 4)
i=2 | giá=1 | k=0:(0, 4)  k=1:(3, 4)  k=2:(3, 4)
i=1 | giá=5 | k=0:(0, 5)  k=1:(3, 8)  k=2:(3, 8)
i=0 | giá=3 | k=0:(0, 5)  k=1:(3, 8)  k=2:(5, 8)
Đáp án dp[0][K][0] = 5


### Độ phức tạp cách 3 và quan sát dẫn tới cách 4

Gọi `n = len(prices)`, `K` = số giao dịch tối đa (ở bài này `K = 2`).

- Thời gian: `O(n·K)` — mỗi ô tính trong `O(1)`; với `K = 2` là `O(n)`.
- Bộ nhớ: `O(n·K)` cho bảng (memo cũng vậy, cộng thêm stack đệ quy `O(n)`).

**Quan sát vàng:** hàng `dp[i]` **chỉ dùng hàng `dp[i+1]`**. Vậy không cần giữ cả bảng — chỉ cần giữ **một hàng**, tức `(K+1) × 2` con số. Với `K = 2`, đó là một nhúm biến. Cách 4 chính là phiên bản "nén" này, nhưng viết theo chiều **xuôi** (từ ngày đầu tới ngày cuối) để dễ hiểu hơn.

## 8. Cách 4 — Bốn biến `buy1, sell1, buy2, sell2` (cách chính)

### 8.1. Kiến thức nền: máy trạng thái (state machine)

Hãy tưởng tượng một người chơi đi qua các ngày, và ở mỗi thời điểm họ đang ở **đúng một** trong 5 "giai đoạn":

```text
            mua           bán            mua           bán
[tay trắng] ----> [cầm lần 1] ----> [đã bán lần 1] ----> [cầm lần 2] ----> [đã bán lần 2]
     ↺ nghỉ           ↺ nghỉ             ↺ nghỉ              ↺ nghỉ             ↺ nghỉ
```

Mỗi mũi tên là một hành động trong một ngày; vòng `↺` là "hôm nay không làm gì". Đây gọi là **máy trạng thái**: một tập trạng thái + luật chuyển giữa chúng. Mọi kế hoạch hợp lệ (0, 1 hay 2 giao dịch, không chồng nhau) đều là **một đường đi** trên sơ đồ này — và sơ đồ **không cho phép** cầm hai cổ phiếu cùng lúc.

### 8.2. Ý nghĩa từng biến: "số dư ví" tốt nhất ở mỗi giai đoạn

Coi như ví bắt đầu với **0 đồng** (được phép âm: mua chịu). Mua giá `p` thì ví `-p`, bán giá `p` thì ví `+p`. Sau khi xử lý xong các ngày `0..i`:

| Biến | Ý nghĩa (số dư ví **lớn nhất** có thể, khi đang ở giai đoạn này) | Ví dụ số |
|---|---|---|
| `buy1` | đã **mua lần 1**, đang cầm. Bằng `-(giá rẻ nhất tới giờ)` | giá rẻ nhất là 3 → `buy1 = -3` |
| `sell1` | đã **bán lần 1** xong. Chính là đáp án bài **#121** trên đoạn `0..i` | mua 3 bán 5 → `sell1 = 2` |
| `buy2` | đã bán lần 1 **rồi mua lần 2**, đang cầm | lãi 2 rồi mua giá 0 → `buy2 = 2 - 0 = 2` |
| `sell2` | đã **bán lần 2** xong (hoặc ít giao dịch hơn) | `buy2 = 2` rồi bán giá 4 → `sell2 = 6` |

Mẹo nhớ: biến `buy*` là "ví đang âm tiền vì vừa ôm hàng", biến `sell*` là "ví sau khi đã chốt lời".

### 8.3. Công thức cập nhật mỗi ngày (giá `p`)

Mỗi biến: hoặc **giữ nguyên** (hôm nay nghỉ, vẫn ở giai đoạn đó), hoặc **đi tới từ giai đoạn trước** bằng hành động hôm nay:

```python
buy1  = max(buy1,  -p)           # nghỉ, hoặc mua lần 1 hôm nay (từ tay trắng, ví 0)
sell1 = max(sell1, buy1 + p)     # nghỉ, hoặc bán lần 1 hôm nay
buy2  = max(buy2,  sell1 - p)    # nghỉ, hoặc mua lần 2 hôm nay (dùng ví sau khi bán lần 1)
sell2 = max(sell2, buy2 + p)     # nghỉ, hoặc bán lần 2 hôm nay
```

**Khởi tạo** (trước ngày đầu tiên):

- `sell1 = sell2 = 0`: chưa giao dịch thì ví 0 (đây cũng là phương án "không giao dịch nào").
- `buy1 = buy2 = float("-inf")`: chưa có ngày nào thì **không thể** đang cầm cổ phiếu. Dùng `-inf` để `max` tự động bỏ qua trạng thái không thể. (Cũng có thể khởi tạo `buy1 = buy2 = -prices[0]`, kết quả như nhau.)

**Đáp án** là `sell2`. Vì sao không phải `max(sell1, sell2)`? Vì `sell2 >= sell1` luôn đúng: "bán lần 1 xong rồi mua lần 2 và bán lần 2 cùng một ngày" cho lãi thêm 0, nên mọi giá trị `sell1` đều "lọt" sang `sell2`. Cell dưới kiểm chứng điều này bằng `assert`.

In [22]:
def max_profit_four_vars(prices: List[int]) -> int:
    """Cách 4 (chính): O(n) thời gian, O(1) bộ nhớ."""
    buy1 = buy2 = float("-inf")    # chưa thể đang cầm cổ phiếu
    sell1 = sell2 = 0              # chưa giao dịch: ví 0
    for p in prices:
        buy1 = max(buy1, -p)
        sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, sell1 - p)
        sell2 = max(sell2, buy2 + p)
        assert sell2 >= sell1      # kiểm chứng nhận xét ở trên
    return sell2

for p, expected in [(ex1, 6), (ex2, 4), (ex3, 0), ([1, 5, 2, 8], 10), ([1, 2, 4, 2, 5, 7, 2, 4, 9, 0], 13)]:
    got = max_profit_four_vars(p)
    print(p, "->", got)
    assert got == expected

[3, 3, 5, 0, 0, 3, 1, 4] -> 6
[1, 2, 3, 4, 5] -> 4
[7, 6, 4, 3, 1] -> 0
[1, 5, 2, 8] -> 10
[1, 2, 4, 2, 5, 7, 2, 4, 9, 0] -> 13


### 8.4. Trace bảng trên Example 1 `[3,3,5,0,0,3,1,4]`

Giá trị **sau khi** cập nhật trong ngày (in đậm là ô vừa thay đổi):

| ngày | giá | buy1 | sell1 | buy2 | sell2 | Chuyện gì xảy ra |
|---:|---:|---:|---:|---:|---:|---|
| 1 | 3 | **-3** | 0 | **-3** | 0 | mua lần 1 giá 3; `buy2 = 0 - 3` (mua lần 2 với ví 0) |
| 2 | 3 | -3 | 0 | -3 | 0 | giá như cũ, không cải thiện gì |
| 3 | 5 | -3 | **2** | -3 | **2** | bán giá 5: lãi 2 (`-3 + 5`); `sell2` cũng nhận 2 |
| 4 | 0 | **0** | 2 | **2** | 2 | giá 0: `buy1 = 0`; `buy2 = sell1 - 0 = 2` — **chốt lãi 2 rồi mua lần 2 giá 0** |
| 5 | 0 | 0 | 2 | 2 | 2 | không đổi |
| 6 | 3 | 0 | **3** | 2 | **5** | `sell1 = 0 + 3` (mua 0 bán 3); `sell2 = 2 + 3` |
| 7 | 1 | 0 | 3 | 2 | 5 | `buy2` có ứng viên `sell1 - 1 = 3 - 1 = 2`, **hòa** với 2 cũ |
| 8 | 4 | 0 | **4** | 2 | **6** | `sell1 = 0 + 4`; `sell2 = 2 + 4 = 6` → **đáp án 6** |

Đọc bảng để hiểu sâu:

- Cột `buy1` luôn bằng `-(giá rẻ nhất tới giờ)`: -3, rồi 0 từ ngày 4. Cột `sell1` chính là trace bài 121.
- Ngày 4 là thời khắc then chốt: `buy2 = 2` nghĩa là "đã ăn trọn ngọn đồi 3→5 (lãi 2), giờ mua lần 2 ở đáy giá 0". Kế hoạch này (B) bán ở ngày 8 được `2 + 4 = 6`.
- Ngày 7 có một **kế hoạch khác** cũng cho `buy2 = 2`: bán lần 1 ở ngày 6 (0→3, lãi 3), mua lần 2 giá 1 → `3 - 1 = 2`. Đó chính là kế hoạch C của đề. Hai kế hoạch hòa nhau, bốn biến không cần biết đã chọn cái nào — chúng chỉ giữ **con số tốt nhất**.

In [23]:
def trace_four_vars(prices):
    buy1 = buy2 = float("-inf")
    sell1 = sell2 = 0
    print("ngày | giá |  buy1 | sell1 |  buy2 | sell2   (* = vừa thay đổi)")
    for day, p in enumerate(prices, start=1):          # đánh số ngày từ 1 như đề
        old = (buy1, sell1, buy2, sell2)
        buy1 = max(buy1, -p)
        sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, sell1 - p)
        sell2 = max(sell2, buy2 + p)
        new = (buy1, sell1, buy2, sell2)
        cells_ = [f"{v:>4}{'*' if v != o else ' '}" for v, o in zip(new, old)]
        print(f"{day:>4} | {p:>3} | " + " | ".join(cells_))
    print("Đáp án sell2 =", sell2)
    return sell2

_ = trace_four_vars(ex1)

ngày | giá |  buy1 | sell1 |  buy2 | sell2   (* = vừa thay đổi)
   1 |   3 |   -3* |    0  |   -3* |    0 
   2 |   3 |   -3  |    0  |   -3  |    0 
   3 |   5 |   -3  |    2* |   -3  |    2*
   4 |   0 |    0* |    2  |    2* |    2 
   5 |   0 |    0  |    2  |    2  |    2 
   6 |   3 |    0  |    3* |    2  |    5*
   7 |   1 |    0  |    3  |    2  |    5 
   8 |   4 |    0  |    4* |    2  |    6*
Đáp án sell2 = 6


### 8.5. Câu hỏi hay: cập nhật "tại chỗ" như vậy có sai không?

Trong một ngày, `sell1` dùng `buy1` **vừa mới cập nhật hôm nay**, `buy2` dùng `sell1` **vừa mới cập nhật**... Nghĩa là ta đang cho phép "mua và bán **cùng một ngày**". Có phạm luật không?

Không sao, vì mọi "hành động cùng ngày" đều **trung hòa**:

- Mua rồi bán cùng ngày giá `p`: lãi `-p + p = 0` → `sell1` nhận ứng viên `buy1 + p` có thể chỉ bằng 0, mà `sell1` vốn `>= 0` rồi.
- Bán lần 1 rồi mua lần 2 cùng ngày: ví như thể **vẫn cầm cổ phiếu** từ lần 1 → tương đương một giao dịch dài hơn, vẫn là kế hoạch hợp lệ.

Tức là: mọi con số mà 4 biến đạt được đều ứng với **một kế hoạch hợp lệ** (không bao giờ "ăn gian"), và mọi kế hoạch hợp lệ đều được xét. Cell dưới kiểm chứng: bản dùng **giá trị cũ của hôm qua** (cập nhật đồng thời bằng tuple) cho kết quả y hệt.

In [24]:
def max_profit_four_vars_simultaneous(prices):
    """Cập nhật ĐỒNG THỜI: vế phải chỉ dùng giá trị của hôm qua."""
    buy1 = buy2 = float("-inf")
    sell1 = sell2 = 0
    for p in prices:
        buy1, sell1, buy2, sell2 = (max(buy1, -p), max(sell1, buy1 + p),
                                    max(buy2, sell1 - p), max(sell2, buy2 + p))
    return sell2

random.seed(1)
for _ in range(2000):
    arr = [random.randint(0, 9) for _ in range(random.randint(1, 12))]
    assert max_profit_four_vars(arr) == max_profit_four_vars_simultaneous(arr)
print("Cập nhật tại chỗ và cập nhật đồng thời: luôn cùng kết quả (2000 test)")

Cập nhật tại chỗ và cập nhật đồng thời: luôn cùng kết quả (2000 test)


### 8.6. Liên hệ cách 3 ↔ cách 4

Cách 3 đi **lùi** và hỏi "từ đây về sau kiếm thêm được bao nhiêu"; cách 4 đi **xuôi** và hỏi "tới giờ ví đã được bao nhiêu". Cả hai dùng **cùng một bộ trạng thái**:

| Cách 3 `(k còn lại, holding)` | Cách 4 |
|---|---|
| `(2, False)` — chưa mua gì | ví 0 (hằng số, không cần biến) |
| `(1, True)` — đã mua lần 1, đang cầm | `buy1` |
| `(1, False)` — đã bán lần 1 | `sell1` |
| `(0, True)` — đã mua lần 2, đang cầm | `buy2` |
| `(0, False)` — đã bán lần 2 | `sell2` |

Đó là lý do chỉ cần **4 biến**: một hàng của bảng DP, bỏ các ô không thể đạt tới.

### 8.7. Vì sao đúng? (invariant bằng lời)

Sau khi xử lý xong ngày `i`, ta duy trì:

- `buy1` = số dư ví lớn nhất trong mọi kế hoạch hợp lệ trên các ngày `0..i` mà **đang cầm cổ phiếu lần 1**.
- `sell1` = số dư lớn nhất của kế hoạch **đã xong tối đa 1 giao dịch**.
- `buy2` = số dư lớn nhất của kế hoạch **đã xong 1 giao dịch và đang cầm cổ phiếu lần 2**.
- `sell2` = số dư lớn nhất của kế hoạch **đã xong tối đa 2 giao dịch**.

Chứng minh theo quy nạp: ngày `i`, một kế hoạch đang ở giai đoạn X thì hoặc **đã ở X từ hôm qua** (ứng viên "giữ nguyên"), hoặc **vừa chuyển vào X hôm nay** từ giai đoạn liền trước (ứng viên "hành động"). Lấy `max` hai ứng viên là bao hết mọi khả năng. Hết mảng, `sell2` chính là đáp án.

### 8.8. Độ phức tạp cách 4

Gọi `n = len(prices)`.

- Thời gian: `O(n)` — một vòng lặp, mỗi ngày 4 phép `max`.
- Bộ nhớ: `O(1)` — đúng 4 biến, không phụ thuộc `n`.

## 9. Code nộp LeetCode

Bản chính dùng cách 4. Code submit gọn, không `print`, không `assert` bên trong. LeetCode tự import `List`; chạy ngoài LeetCode ta đã import ở đầu notebook.

In [25]:
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        buy1 = buy2 = float("-inf")
        sell1 = sell2 = 0
        for p in prices:
            buy1 = max(buy1, -p)
            sell1 = max(sell1, buy1 + p)
            buy2 = max(buy2, sell1 - p)
            sell2 = max(sell2, buy2 + p)
        return sell2


sol = Solution()
# Ví dụ của đề
assert sol.maxProfit([3, 3, 5, 0, 0, 3, 1, 4]) == 6
assert sol.maxProfit([1, 2, 3, 4, 5]) == 4
assert sol.maxProfit([7, 6, 4, 3, 1]) == 0
# Edge case
assert sol.maxProfit([5]) == 0                       # 1 ngày: không giao dịch được
assert sol.maxProfit([1, 2]) == 1
assert sol.maxProfit([2, 1]) == 0
assert sol.maxProfit([4, 4, 4, 4]) == 0              # giá phẳng
assert sol.maxProfit([0, 100000]) == 100000          # giá biên theo constraints
assert sol.maxProfit([1, 5, 2, 8]) == 10             # tách đôi tốt hơn 1 giao dịch dài
assert sol.maxProfit([1, 2, 4, 2, 5, 7, 2, 4, 9, 0]) == 13   # phải gộp đoạn tăng
assert sol.maxProfit([1, 3, 1, 3, 1, 3]) == 4        # 3 cơ hội nhưng chỉ được lấy 2
print("class Solution: PASS toàn bộ test!")

class Solution: PASS toàn bộ test!


Một bản nộp khác (cách 2, prefix/suffix) — cũng `O(n)`, hữu ích nếu bạn thấy "chia mốc" dễ nhớ hơn. Ở đây đặt tên class khác để không ghi đè bản chính; khi nộp thì đổi lại thành `Solution`.

In [26]:
class SolutionPrefixSuffix:
    def maxProfit(self, prices: List[int]) -> int:
        n = len(prices)
        left = [0] * n
        lo = prices[0]
        for i in range(1, n):
            lo = min(lo, prices[i])
            left[i] = max(left[i - 1], prices[i] - lo)
        best, hi, right = left[-1], prices[-1], 0
        for i in range(n - 2, -1, -1):          # gộp: vừa dựng right vừa lấy max
            hi = max(hi, prices[i])
            right = max(right, hi - prices[i])
            best = max(best, left[i] + right)
        return best

s2 = SolutionPrefixSuffix()
for p, e in [([3, 3, 5, 0, 0, 3, 1, 4], 6), ([1, 2, 3, 4, 5], 4), ([7, 6, 4, 3, 1], 0), ([5], 0), ([1, 5, 2, 8], 10)]:
    assert s2.maxProfit(p) == e
print("SolutionPrefixSuffix: PASS")

SolutionPrefixSuffix: PASS


### Kiểm thử đối chiếu chéo tất cả các cách

Ta sinh nhiều mảng ngẫu nhiên nhỏ (cố định `random.seed` để lần chạy nào cũng giống nhau) và so **mọi cách** với vét cạn `O(n⁴)`. Nếu có cách nào sai dù chỉ một ca, `assert` sẽ báo ngay kèm input.

Sau đó chạy hai cách `O(n)` trên mảng đúng cỡ constraints `n = 10^5`.

In [27]:
import io, contextlib

random.seed(123)
for _ in range(600):
    arr = [random.randint(0, 8) for _ in range(random.randint(1, 9))]
    expected = max_profit_bruteforce(arr)
    with contextlib.redirect_stdout(io.StringIO()):          # tắt dòng in của max_profit_memo
        memo_ans = max_profit_memo(arr)
    results = {
        "split": max_profit_split(arr),
        "prefix_suffix": max_profit_prefix_suffix(arr),
        "memo": memo_ans,
        "dp_table": max_profit_dp_table(arr),
        "four_vars": max_profit_four_vars(arr),
        "Solution": Solution().maxProfit(arr),
        "SolutionPrefixSuffix": SolutionPrefixSuffix().maxProfit(arr),
    }
    for name, got in results.items():
        assert got == expected, (name, arr, got, expected)
print("600 mảng ngẫu nhiên: mọi cách khớp vét cạn O(n^4)")

big = [random.randint(0, 10**5) for _ in range(10**5)]
t0 = time.perf_counter(); a1 = Solution().maxProfit(big); t1 = time.perf_counter()
a2 = SolutionPrefixSuffix().maxProfit(big); t2 = time.perf_counter()
assert a1 == a2 == max_profit_dp_table(big)
print(f"n = 10^5: four_vars {t1 - t0:.3f}s, prefix_suffix {t2 - t1:.3f}s, đáp án = {a1}")

600 mảng ngẫu nhiên: mọi cách khớp vét cạn O(n^4)


n = 10^5: four_vars 0.013s, prefix_suffix 0.019s, đáp án = 199994


## 10. So sánh các cách

Gọi `n = len(prices)`, `K` = số giao dịch tối đa (bài này `K = 2`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| Vét cạn 4 vòng | thử mọi `i <= j <= k <= l` | `O(n⁴)` | `O(1)` | chỉ làm "trọng tài" để test |
| 1. Chia mốc | thử mọi mốc, mỗi bên giải #121 | `O(n²)` | `O(n)` | dễ nghĩ, quá chậm với `n = 10^5` |
| 2. Prefix / Suffix | tính sẵn `left[i]`, `right[i]` | `O(n)` | `O(n)` | nộp được; chỉ hợp với đúng 2 giao dịch |
| 3a. DP memo | đệ quy trạng thái `(i, k, holding)` | `O(n·K)` | `O(n·K)` | dễ suy ra; `RecursionError` khi `n` lớn |
| 3b. DP bảng | điền bảng từ ngày cuối lùi về | `O(n·K)` | `O(n·K)` | tổng quát cho #188 |
| **4. Bốn biến** | nén một hàng DP, đi xuôi | **`O(n)`** | **`O(1)`** | **cách chính**, ngắn nhất |

## 11. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo `buy1 = buy2 = 0`

`buy1 = 0` nghĩa là "đang cầm cổ phiếu mà **không tốn đồng nào**" — một trạng thái không có thật. Với `[3,5]`: `sell1 = max(0, 0 + 5) = 5`, sai (đúng là 2). **Sửa:** khởi tạo `float("-inf")` (hoặc `-prices[0]`).

### Lỗi 2: Quên cộng `sell1` khi mua lần 2: `buy2 = max(buy2, -p)`

Khi đó `buy2` giống hệt `buy1`, giao dịch 2 "quên" mất lãi của giao dịch 1 → đáp án chỉ bằng bài #121. Với `[1,5,2,8]` ra 7 thay vì 10. **Sửa:** `buy2 = max(buy2, sell1 - p)`.

### Lỗi 3: Cách 2 lấy `max(left) + max(right)`

Hai giá trị max có thể đến từ **hai giao dịch chồng nhau** (cùng là giao dịch 1→8 trong `[1,5,2,8]`) → ra 14, vượt cả mức có thể. **Sửa:** cộng **cùng một mốc**: `max(left[k] + right[k])`.

### Lỗi 4: Vét cạn cho mua lần 2 **trước** khi bán lần 1

Vòng lặp `for k in range(n)` thay vì `range(j, n)` → cho phép hai giao dịch chồng nhau, dùng một đợt tăng giá hai lần. `[1,5,2,8]` ra 14. **Sửa:** ngày mua lần 2 `k` phải `>= j`.

### Lỗi 5: Nộp bản đệ quy memo với `n = 10^5`

Độ sâu đệ quy bằng `n` → `RecursionError`. **Sửa:** dùng bảng bottom-up hoặc 4 biến. (Tăng `sys.setrecursionlimit` là cách "chữa cháy" dễ làm sập chương trình, không nên.)

Cell dưới chạy từng phiên bản sai để bạn thấy tận mắt.

In [28]:
def wrong_init_zero(prices):                 # Lỗi 1
    buy1 = buy2 = 0
    sell1 = sell2 = 0
    for p in prices:
        buy1 = max(buy1, -p); sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, sell1 - p); sell2 = max(sell2, buy2 + p)
    return sell2

def wrong_forget_sell1(prices):              # Lỗi 2
    buy1 = buy2 = float("-inf")
    sell1 = sell2 = 0
    for p in prices:
        buy1 = max(buy1, -p); sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, -p); sell2 = max(sell2, buy2 + p)    # SAI: thiếu sell1
    return sell2

def wrong_max_plus_max(prices):              # Lỗi 3
    return max(build_left(prices)) + max(build_right(prices))

def wrong_overlap_bruteforce(prices):        # Lỗi 4
    n, best = len(prices), 0
    for i in range(n):
        for j in range(i, n):
            for k in range(n):               # SAI: phải là range(j, n)
                for l in range(k, n):
                    best = max(best, prices[j] - prices[i] + prices[l] - prices[k])
    return best

print("[3,5]     đúng:", Solution().maxProfit([3, 5]), "| lỗi 1 (buy=0):", wrong_init_zero([3, 5]))
p = [1, 5, 2, 8]
print("[1,5,2,8] đúng:", Solution().maxProfit(p),
      "| lỗi 2:", wrong_forget_sell1(p), "| lỗi 3:", wrong_max_plus_max(p), "| lỗi 4:", wrong_overlap_bruteforce(p))

try:                                         # Lỗi 5
    with contextlib.redirect_stdout(io.StringIO()):
        max_profit_memo(list(range(10**5)))
except RecursionError as e:
    print("lỗi 5 (memo với n = 10^5): RecursionError -", e)

[3,5]     đúng: 2 | lỗi 1 (buy=0): 5
[1,5,2,8] đúng: 10 | lỗi 2: 7 | lỗi 3: 14 | lỗi 4: 14
lỗi 5 (memo với n = 10^5): RecursionError - maximum recursion depth exceeded


## 12. Mẹo nhận dạng pattern

- Đề **mua/bán**, **có giới hạn số lần** (tối đa `k` lần), **không giữ hai thứ cùng lúc** → nghĩ ngay tới **DP trạng thái**: liệt kê các "giai đoạn" (tay trắng / đang cầm lần 1 / đã bán lần 1 / ...), mỗi ngày mỗi giai đoạn chọn "giữ nguyên" hoặc "chuyển từ giai đoạn trước".
- Khi bảng DP mà **hàng `i` chỉ phụ thuộc hàng `i-1` (hoặc `i+1`)** → nén còn vài biến, bộ nhớ `O(1)`.
- Đề "**hai** đối tượng không chồng nhau, tối ưu tổng" (hai giao dịch, hai đoạn con, hai khoảng) → thử **chia mốc + prefix/suffix**: `best = max(left[i] + right[i])` hoặc `max(left[i] + right[i+1])`.
- Khi mẹo tham lam cho kết quả lệch so với vét cạn trên mảng nhỏ → chắc chắn cần DP. Luôn viết một bộ vét cạn nhỏ để kiểm tra!

## 13. Biến thể và bài liên quan

| Bài | Ràng buộc | Ý tưởng |
|---|---|---|
| #121 Best Time to Buy and Sell Stock | 1 giao dịch | running min (mục 3) |
| #122 Best Time to Buy and Sell Stock II | không giới hạn số giao dịch | cộng mọi đoạn tăng `max(0, p[i] - p[i-1])` |
| **#123 (bài này)** | **tối đa 2** | 4 biến / prefix-suffix |
| **#188 Best Time to Buy and Sell Stock IV** | **tối đa `k`** | tổng quát hoá: mảng `buy[1..k]`, `sell[1..k]` (Bài 6) |
| #309 ... with Cooldown | bán xong phải nghỉ 1 ngày | máy trạng thái thêm giai đoạn "đang nghỉ" |
| #714 ... with Transaction Fee | mỗi giao dịch mất phí | trừ phí khi bán (Bài 5 là phiên bản 2 giao dịch + phí) |

**Liên hệ #188:** bài này là #188 với `k = 2`. Hàm `max_profit_dp_table(prices, K)` ở cách 3 đã giải được #188 với `O(n·k)` bộ nhớ. Bài 6 yêu cầu bạn viết phiên bản `O(k)` bộ nhớ bằng cách mở rộng 4 biến thành hai mảng. Mẹo thêm: nếu `k >= n // 2` thì giới hạn `k` coi như không có (không thể làm quá `n // 2` giao dịch có lãi), bài trở về #122.

In [29]:
# Kiểm chứng nhanh nhận xét "k lớn thì về bài 122" bằng chính bảng DP của cách 3
def max_profit_unlimited(prices):            # bài 122
    return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, len(prices)))

p = [1, 3, 1, 3, 1, 3]
for K in range(1, 5):
    print(f"K={K}: {max_profit_dp_table(p, K)}")
print("Bài 122 (không giới hạn):", max_profit_unlimited(p))

K=1: 2
K=2: 4
K=3: 6
K=4: 6
Bài 122 (không giới hạn): 6


## 14. Bài tập vận dụng

Làm theo thứ tự từ dễ tới khó. Gợi ý ở mục 15, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay bảng 4 biến (không chạy code)

Lập bảng `buy1, sell1, buy2, sell2` sau mỗi ngày cho `prices = [2,1,4,5,2,9,7]`. Đáp án là bao nhiêu? Kế hoạch giao dịch nào đạt được con số đó?

### Bài 2 — Đoán output

Không chạy code, hãy đoán:

- (a) `build_left([3,2,6,5,0,3])` và `build_right([3,2,6,5,0,3])`, rồi đáp án cách 2.
- (b) `wrong_init_zero([1,5,2,8])` (Lỗi 1, khởi tạo `buy = 0`) trả về bao nhiêu? Vì sao lại lớn hơn đáp án đúng 10?

### Bài 3 — Điền TODO: cách prefix/suffix

Hoàn thiện hàm theo 3 bước: dựng `left` (đi xuôi, nhớ giá **thấp** nhất), dựng `right` (đi ngược, nhớ giá **cao** nhất), lấy `max(left[k] + right[k])`.

In [30]:
def exercise_prefix_suffix(prices: List[int]) -> int:
    n = len(prices)
    left = [0] * n
    right = [0] * n
    # TODO 1: đi xuôi i = 1..n-1, cập nhật min_price và left[i]
    # TODO 2: đi ngược i = n-2..0, cập nhật max_price và right[i]
    # TODO 3: trả về max(left[k] + right[k]) với mọi k
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_prefix_suffix([3, 3, 5, 0, 0, 3, 1, 4]) == 6
# assert exercise_prefix_suffix([1, 2, 3, 4, 5]) == 4
# assert exercise_prefix_suffix([7, 6, 4, 3, 1]) == 0
# assert exercise_prefix_suffix([5]) == 0

### Bài 4 — Tự viết lại cách 4 từ trang giấy trắng

Không nhìn lại mục 8, viết hàm 4 biến. Trước khi code, hãy tự trả lời bằng lời:

1. Mỗi biến `buy1, sell1, buy2, sell2` nghĩa là gì?
2. Vì sao `buy` khởi tạo `-inf` còn `sell` khởi tạo `0`?
3. Trả về biến nào? Vì sao không cần `max(sell1, sell2)`?

Sau đó, thử phiên bản khởi tạo `buy1 = buy2 = -prices[0]` và kiểm tra hai phiên bản cho cùng kết quả.

In [31]:
def exercise_four_vars(prices: List[int]) -> int:
    # TODO: khởi tạo 4 biến
    # TODO: với mỗi giá p, cập nhật buy1 -> sell1 -> buy2 -> sell2
    # TODO: trả về đáp án
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_four_vars([3, 3, 5, 0, 0, 3, 1, 4]) == 6
# assert exercise_four_vars([1, 2, 3, 4, 5]) == 4
# assert exercise_four_vars([7, 6, 4, 3, 1]) == 0
# assert exercise_four_vars([1, 2, 4, 2, 5, 7, 2, 4, 9, 0]) == 13

### Bài 5 — Biến thể: tối đa 2 giao dịch **có phí**

Vẫn tối đa 2 giao dịch, nhưng **mỗi giao dịch** (một lần mua + một lần bán) bị trừ phí `fee`. Giao dịch nào lãi không đủ bù phí thì tốt nhất đừng làm. Viết `max_profit_two_with_fee(prices, fee)` trong `O(n)` thời gian, `O(1)` bộ nhớ.

```python
max_profit_two_with_fee([1,3,2,8,4,9], 2)       -> 8    # (1→8: 7-2) + (4→9: 5-2)
max_profit_two_with_fee([3,3,5,0,0,3,1,4], 1)   -> 4
max_profit_two_with_fee([1,2], 5)               -> 0
```

In [32]:
def exercise_two_with_fee(prices: List[int], fee: int) -> int:
    # TODO: giống 4 biến, nhưng trừ fee ở đúng một chỗ cho mỗi giao dịch
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_two_with_fee([1, 3, 2, 8, 4, 9], 2) == 8
# assert exercise_two_with_fee([1, 3, 7, 5, 10, 3], 3) == 6
# assert exercise_two_with_fee([3, 3, 5, 0, 0, 3, 1, 4], 1) == 4
# assert exercise_two_with_fee([1, 2], 5) == 0

### Bài 6 — Mini challenge: LeetCode #188 — tối đa `k` giao dịch

Signature của LeetCode #188:

```python
class Solution:
    def maxProfit(self, k: int, prices: List[int]) -> int:
```

Ví dụ của đề #188: `k = 2, prices = [2,4,1]` → `2`; `k = 2, prices = [3,2,6,5,0,3]` → `7`.

Yêu cầu: tổng quát hoá 4 biến thành **hai mảng** `buy[1..k]`, `sell[0..k]` (với `sell[0] = 0` là "ví lúc tay trắng"), thời gian `O(n·k)`, bộ nhớ `O(k)`. Thêm tối ưu: nếu `k >= n // 2` thì trả về đáp án bài #122.

In [33]:
def exercise_max_profit_k(k: int, prices: List[int]) -> int:
    # TODO 1: nếu k >= len(prices) // 2: cộng mọi đoạn tăng (bài 122)
    # TODO 2: buy = [-inf] * (k + 1), sell = [0] * (k + 1)
    # TODO 3: với mỗi p, với j = 1..k: buy[j] = max(buy[j], sell[j-1] - p); sell[j] = max(sell[j], buy[j] + p)
    # TODO 4: trả về sell[k]
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_profit_k(2, [2, 4, 1]) == 2
# assert exercise_max_profit_k(2, [3, 2, 6, 5, 0, 3]) == 7
# assert exercise_max_profit_k(2, [3, 3, 5, 0, 0, 3, 1, 4]) == 6
# assert exercise_max_profit_k(1, [7, 1, 5, 3, 6, 4]) == 5

## 15. Gợi ý theo mức

**Bài 1**
- Mức 1: làm y như bảng ở mục 8.4, mỗi ngày cập nhật theo thứ tự `buy1 → sell1 → buy2 → sell2`.
- Mức 2: giá rẻ nhất là 1 (ngày 2), nên từ ngày 2 trở đi `buy1 = -1`.
- Mức 3: ngày 5 (giá 2) là lúc `buy2` nhảy lên: `sell1 - 2 = 4 - 2 = 2`.

**Bài 2**
- Mức 1: `left` = đáp án bài 121 trên đoạn `0..i`; `right` = đáp án bài 121 trên đoạn `i..n-1`.
- Mức 2: (a) giá thấp nhất bên trái đổi từ 3 → 2 → 0; giá cao nhất bên phải đổi từ 3 → 5 → 6.
- Mức 3: (b) với `buy1 = 0`, ngay ngày đầu `sell1 = max(0, 0 + 1) = 1` — "bán" một cổ phiếu chưa từng mua.

**Bài 3**
- Mức 1: chép lại hai hàm `build_left`, `build_right` của mục 6 nhưng tự gõ.
- Mức 2: vòng ngược viết `for i in range(n - 2, -1, -1):`.
- Mức 3: `left[i] = max(left[i-1], prices[i] - min_price)`, `right[i] = max(right[i+1], max_price - prices[i])`.

**Bài 4**
- Mức 1: nghĩ "số dư ví" ở mỗi giai đoạn của máy trạng thái.
- Mức 2: `buy` là trạng thái đang cầm → chưa có ngày nào thì không thể cầm → `-inf`.
- Mức 3: `sell2 >= sell1` vì có thể mua-bán lần 2 cùng một ngày (lãi thêm 0).

**Bài 5**
- Mức 1: phí gắn với **một giao dịch**, nên chỉ trừ **một lần** mỗi cặp mua-bán (khi mua hoặc khi bán, chọn một).
- Mức 2: ví dụ trừ khi bán: `sell1 = max(sell1, buy1 + p - fee)`.
- Mức 3: tương tự `sell2 = max(sell2, buy2 + p - fee)`; hai dòng `buy` giữ nguyên.

**Bài 6**
- Mức 1: `buy1, buy2` → `buy[1], buy[2]`; `sell1, sell2` → `sell[1], sell[2]`; ví "tay trắng" 0 → `sell[0]`.
- Mức 2: công thức `buy[j] = max(buy[j], sell[j-1] - p)` là bản tổng quát của `buy2 = max(buy2, sell1 - p)`.
- Mức 3: với mỗi `p`, cho `j` chạy `1..k` theo thứ tự tăng (cập nhật tại chỗ vẫn đúng như mục 8.5).

## 16. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1** — `prices = [2,1,4,5,2,9,7]`

| ngày | giá | buy1 | sell1 | buy2 | sell2 |
|---:|---:|---:|---:|---:|---:|
| 1 | 2 | -2 | 0 | -2 | 0 |
| 2 | 1 | -1 | 0 | -1 | 0 |
| 3 | 4 | -1 | 3 | -1 | 3 |
| 4 | 5 | -1 | 4 | -1 | 4 |
| 5 | 2 | -1 | 4 | 2 | 4 |
| 6 | 9 | -1 | 8 | 2 | **11** |
| 7 | 7 | -1 | 8 | 2 | 11 |

Đáp án **11**: mua giá 1 (ngày 2) bán giá 5 (ngày 4), lãi 4; mua giá 2 (ngày 5) bán giá 9 (ngày 6), lãi 7. Để ý `sell1 = 8` (mua 1 bán 9) là giao dịch đơn tốt nhất, nhưng tách đôi được 11.

**Bài 2**

- (a) `left = [0,0,4,4,4,4]`, `right = [4,4,3,3,3,0]`, `left + right = [4,4,7,7,7,4]` → đáp án **7** (mua 2 bán 6, mua 0 bán 3).
- (b) `wrong_init_zero([1,5,2,8])` trả về **11**. Vì `buy1 = 0` cho phép "bán không cần mua": ngay ngày 1, `sell1 = 0 + 1 = 1`, rồi `buy2 = 1 - 1 = 0`... lãi ảo cứ thế cộng dồn. Kết quả lớn hơn đáp án đúng là dấu hiệu chắc chắn của trạng thái "không có thật".

Cell dưới kiểm chứng bài 1, bài 2 bằng máy, sau đó là đáp án code bài 3–6.

In [34]:
# Kiểm chứng Bài 1 và Bài 2
assert trace_four_vars([2, 1, 4, 5, 2, 9, 7]) == 11
print()
assert build_left([3, 2, 6, 5, 0, 3]) == [0, 0, 4, 4, 4, 4]
assert build_right([3, 2, 6, 5, 0, 3]) == [4, 4, 3, 3, 3, 0]
assert max_profit_prefix_suffix([3, 2, 6, 5, 0, 3]) == 7
assert wrong_init_zero([1, 5, 2, 8]) == 11
print("Bài 2: PASS")

ngày | giá |  buy1 | sell1 |  buy2 | sell2   (* = vừa thay đổi)
   1 |   2 |   -2* |    0  |   -2* |    0 
   2 |   1 |   -1* |    0  |   -1* |    0 
   3 |   4 |   -1  |    3* |   -1  |    3*
   4 |   5 |   -1  |    4* |   -1  |    4*
   5 |   2 |   -1  |    4  |    2* |    4 
   6 |   9 |   -1  |    8* |    2  |   11*
   7 |   7 |   -1  |    8  |    2  |   11 
Đáp án sell2 = 11

Bài 2: PASS


In [35]:
def answer_prefix_suffix(prices: List[int]) -> int:
    n = len(prices)
    left = [0] * n
    right = [0] * n
    min_price = prices[0]
    for i in range(1, n):
        min_price = min(min_price, prices[i])
        left[i] = max(left[i - 1], prices[i] - min_price)
    max_price = prices[-1]
    for i in range(n - 2, -1, -1):
        max_price = max(max_price, prices[i])
        right[i] = max(right[i + 1], max_price - prices[i])
    return max(left[k] + right[k] for k in range(n))


def answer_four_vars(prices: List[int]) -> int:
    buy1 = buy2 = float("-inf")
    sell1 = sell2 = 0
    for p in prices:
        buy1 = max(buy1, -p)
        sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, sell1 - p)
        sell2 = max(sell2, buy2 + p)
    return sell2


def answer_four_vars_init_first(prices: List[int]) -> int:
    buy1 = buy2 = -prices[0]          # "mua ngay ngày đầu" làm giá trị khởi đầu
    sell1 = sell2 = 0
    for p in prices[1:]:
        buy1 = max(buy1, -p)
        sell1 = max(sell1, buy1 + p)
        buy2 = max(buy2, sell1 - p)
        sell2 = max(sell2, buy2 + p)
    return sell2


def answer_two_with_fee(prices: List[int], fee: int) -> int:
    buy1 = buy2 = float("-inf")
    sell1 = sell2 = 0
    for p in prices:
        buy1 = max(buy1, -p)
        sell1 = max(sell1, buy1 + p - fee)    # trừ phí khi bán lần 1
        buy2 = max(buy2, sell1 - p)
        sell2 = max(sell2, buy2 + p - fee)    # trừ phí khi bán lần 2
    return sell2


def answer_max_profit_k(k: int, prices: List[int]) -> int:
    n = len(prices)
    if k >= n // 2:                           # coi như không giới hạn: bài 122
        return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, n))
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)                      # sell[0] = 0: ví lúc tay trắng
    for p in prices:
        for j in range(1, k + 1):
            buy[j] = max(buy[j], sell[j - 1] - p)
            sell[j] = max(sell[j], buy[j] + p)
    return sell[k]


assert answer_prefix_suffix([3, 3, 5, 0, 0, 3, 1, 4]) == 6
assert answer_prefix_suffix([1, 2, 3, 4, 5]) == 4
assert answer_prefix_suffix([7, 6, 4, 3, 1]) == 0
assert answer_prefix_suffix([5]) == 0
assert answer_four_vars([1, 2, 4, 2, 5, 7, 2, 4, 9, 0]) == 13
assert answer_two_with_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert answer_two_with_fee([1, 3, 7, 5, 10, 3], 3) == 6
assert answer_two_with_fee([3, 3, 5, 0, 0, 3, 1, 4], 1) == 4
assert answer_two_with_fee([1, 2], 5) == 0
assert answer_max_profit_k(2, [2, 4, 1]) == 2
assert answer_max_profit_k(2, [3, 2, 6, 5, 0, 3]) == 7
assert answer_max_profit_k(2, [3, 3, 5, 0, 0, 3, 1, 4]) == 6
assert answer_max_profit_k(1, [7, 1, 5, 3, 6, 4]) == 5


def bruteforce_two_with_fee(prices, fee):     # trọng tài cho Bài 5
    n, best = len(prices), 0
    for i in range(n):
        for j in range(i, n):
            first = prices[j] - prices[i] - fee
            best = max(best, first)
            for k in range(j, n):
                for l in range(k, n):
                    best = max(best, first + prices[l] - prices[k] - fee)
    return best


# Đối chiếu ngẫu nhiên các đáp án với vét cạn / bảng DP
random.seed(2024)
for _ in range(400):
    arr = [random.randint(0, 9) for _ in range(random.randint(1, 9))]
    ref = max_profit_bruteforce(arr)
    assert answer_prefix_suffix(arr) == ref
    assert answer_four_vars(arr) == ref
    assert answer_four_vars_init_first(arr) == ref
    fee = random.randint(0, 4)
    assert answer_two_with_fee(arr, fee) == bruteforce_two_with_fee(arr, fee)
    kk = random.randint(1, 5)
    assert answer_max_profit_k(kk, arr) == max_profit_dp_table(arr, kk)
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 17. Template ngắn nên nhớ

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        buy1 = buy2 = float("-inf")    # đang cầm lần 1 / lần 2: ban đầu "không thể"
        sell1 = sell2 = 0              # đã bán xong lần 1 / lần 2: ban đầu ví 0
        for p in prices:
            buy1  = max(buy1,  -p)          # mua lần 1 (từ ví 0)
            sell1 = max(sell1, buy1 + p)    # bán lần 1
            buy2  = max(buy2,  sell1 - p)   # mua lần 2 (dùng ví sau lần 1)
            sell2 = max(sell2, buy2 + p)    # bán lần 2
        return sell2
```

Câu thần chú: **"Mỗi giai đoạn: giữ nguyên, hoặc đến từ giai đoạn ngay trước. Mua thì trừ giá, bán thì cộng giá."**

Tổng quát cho `k` giao dịch (#188): `buy[j] = max(buy[j], sell[j-1] - p)`, `sell[j] = max(sell[j], buy[j] + p)`, với `sell[0] = 0`.

## 18. Checklist trước khi submit

- `buy1`, `buy2` khởi tạo `float("-inf")` (hoặc `-prices[0]`), **không phải 0**?
- `buy2` dùng `sell1 - p` (mang lãi lần 1 sang), không phải `-p`?
- Trả về `sell2` (luôn `>= sell1`, đã bao gồm phương án 0 hay 1 giao dịch)?
- Lợi nhuận không bao giờ âm: mảng giảm dần phải ra 0?
- Đã thử: 1 ngày, 2 ngày tăng/giảm, giá phẳng, 3 ví dụ của đề, và ca "tách đôi tốt hơn" `[1,5,2,8] → 10`?
- Không dùng đệ quy sâu `n` tầng khi `n` có thể tới `10^5`?
- Nếu dùng prefix/suffix: cộng `left[k] + right[k]` **cùng mốc**, không phải `max(left) + max(right)`?